# Delivery count prediction (exploratory notebook)

Typical sequential data science code. It works, but how do we test it?
See `delivery/` for the same code restructured into testable functions.

In [ ]:
test_size = 0.3
random_state = 1
data_path = '../data/train.csv'

In [ ]:
import pandas as pd
from scipy.stats import boxcox
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv(data_path, parse_dates=True)
df.head()

In [ ]:
# data cleaning
df = df.drop_duplicates()
df = df[df['delivery_count'] > 0]
df = df[df['humidity'] <= 100]
df.describe()

In [ ]:
# feature engineering
df['month'] = pd.to_datetime(df['datetime']).dt.month
df['dayofweek'] = pd.to_datetime(df['datetime']).dt.dayofweek
df['hour'] = pd.to_datetime(df['datetime']).dt.hour
df['delivery_count'] = boxcox(df['delivery_count'], 0.4)
df.drop(['datetime'], axis=1, inplace=True)

dummies = pd.get_dummies(df, columns=['month', 'weather', 'dayofweek', 'hour'])
dummies = dummies.drop(['month_1', 'hour_0', 'weather_1'], axis=1)

X = dummies.drop(['delivery_count'], axis=1)
y = pd.Series(df['delivery_count'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)

In [ ]:
# training and evaluation
lr = LinearRegression()
lr.fit(X_train, y_train)

print(lr.score(X_train, y_train))
print(lr.score(X_test, y_test))